# HeartlyPE — CV reproducible en Colab (GPU)

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.

Corre las celdas en orden. Los resultados se guardan en **Google Drive** (`MyDrive/heartlype_runs/`), así que si Colab se desconecta basta con volver a correr todo: los jobs (semilla × fold) ya terminados se saltan.

## 1. Clonar el código y montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/ecg-arrhythmia-thesis
!git clone -q -b data-artifacts https://github.com/ClaudiaSifuentes/ecg-arrhythmia-thesis.git /content/ecg-arrhythmia-thesis
%cd /content/ecg-arrhythmia-thesis
!git log --oneline -1

## 2. Dependencias

In [ ]:
!pip -q install wfdb neurokit2 pyyaml scikit-learn
import torch; print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU (revisa el tipo de entorno)')

## 3. Datos MIT-BIH + INCART (desde la rama `data-incart`, vía Git LFS)

In [ ]:
# Colab no trae Git LFS: sin esto los .npy bajan como "punteros" de ~130 bytes
!apt-get -qq install -y git-lfs > /dev/null && git lfs install --skip-repo
!rm -rf /content/data_incart
!git clone -q --depth 1 -b data-incart https://github.com/ClaudiaSifuentes/ecg-arrhythmia-thesis.git /content/data_incart
!mkdir -p data/processed/mitbih_incart && cp /content/data_incart/data/processed/mitbih_incart/*.npy data/processed/mitbih_incart/
!ls -la data/processed/mitbih_incart/
import os
size = os.path.getsize('data/processed/mitbih_incart/X_beats.npy')
assert size > 200_000_000, f'X_beats.npy pesa {size} bytes: LFS no descargó los datos. Vuelve a correr esta celda.'
print('OK: datos reales descargados')

## 4. Salidas persistentes en Drive
Los resultados (`reports/cv`, `reports/oof`, `reports/thresholds`, `reports/test`, `models/cv`, `models/final`) se enlazan a Drive.

In [ ]:
import os
RUNS = '/content/drive/MyDrive/heartlype_runs'
for d in ['reports/cv', 'reports/oof', 'reports/thresholds', 'reports/test', 'models/cv', 'models/final']:
    target = f'{RUNS}/{d}'
    os.makedirs(target, exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        !cp -rn {d}/. {target}/ 2>/dev/null; rm -rf {d}
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); os.symlink(target, d)
!ls -la reports models

## 5. Mapa registro → paciente de INCART
Ya viene en el repo (`configs/incart_subjects.json`, 75 registros → 32 pacientes, leído del campo `patient N` de los encabezados). Esta celda solo lo verifica.

In [ ]:
import json
m = json.load(open('configs/incart_subjects.json'))['subject_of_record']
print(len(m), 'registros INCART ->', len(set(m.values())), 'pacientes')   # debe decir 75 -> 32

## 6. Tests de determinismo (ahora en GPU)

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_protocol.py

## 7. Baseline oficial E12 (3 semillas × 5 folds)
Receta E12 sin cambios; early stopping con validación interna por paciente; umbral SVEB 0.24 del config aplicado igual en CV y test.

In [ ]:
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E12.yaml

## 8. Barrido de umbrales sobre OOF y variantes post-hoc (ensamble / temperatura)

In [ ]:
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E12
!PYTHONPATH=. python scripts/posthoc_variants.py --exp E12

## 9. Experimento E13: contexto RR (un solo cambio)

In [ ]:
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E13_rrctx.yaml
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E13_rrctx

## 11. E14 — INCART reconstruido con derivación II (un solo cambio sobre E13)
**Si vienes de una sesión nueva:** corre primero las secciones 1 a 6 y luego salta aquí (las secciones 7–9 ya están hechas y guardadas en Drive).

Los datos construidos se guardan en Drive (`heartlype_runs/data/`), así que solo se construyen una vez. Construir INCART tarda ~15–30 min (detección de picos en 64 registros).

In [ ]:
import os
RUNS = '/content/drive/MyDrive/heartlype_runs'
for d in ['data/processed/inc_ch1', 'data/processed/mitbih_incartII', 'data/processed/svdb_ch0']:
    target = f'{RUNS}/{d}'
    os.makedirs(target, exist_ok=True)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); os.symlink(target, d)

if not os.path.exists('data/processed/inc_ch1/X_beats.npy'):
    !PYTHONPATH=. python scripts/build_extra_db.py --db incartdb --name inc --channel 1 --fs 257
if not os.path.exists('data/processed/mitbih_incartII/X_beats.npy'):
    !PYTHONPATH=. python scripts/build_extra_db.py --merge data/processed/mitbih_incart data/processed/inc_ch1 --keep_prefix mit_ inc_ --out data/processed/mitbih_incartII

In [ ]:
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E14_incII.yaml
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E14_incII

## 12. Construir SVDB (MIT-BIH Supraventricular Arrhythmia DB) — solo datos, para el experimento E15
78 registros a 128 Hz con muchos SVEB. Aquí solo se construye y se guarda en Drive; el experimento se decide después de ver E14.

In [ ]:
if not os.path.exists('data/processed/svdb_ch0/X_beats.npy'):
    !PYTHONPATH=. python scripts/build_extra_db.py --db svdb --name svdb --channel 0 --fs 128
import numpy as np
y = np.load('data/processed/svdb_ch0/y_beats.npy'); pid = np.load('data/processed/svdb_ch0/patient_id.npy')
print('SVDB:', len(y), 'latidos,', len(set(pid)), 'registros, N/SVEB/VEB =', np.bincount(y, minlength=3))

## 13. Test fijo (6 pacientes) — E12 vs E13 (y E14 si ya corrió)
Entrena cada receta con los pacientes de train (MIT-BIH train + INCART) y early stopping con los 6 pacientes de val, 3 semillas, y reporta el test **completo y sin el registro 232**, con argmax y el ensamble de las 3 semillas. El test solo se reporta: no se usa para elegir nada. ~5–10 min por receta en GPU.

In [ ]:
!PYTHONPATH=. python scripts/eval_test.py --config configs/experiments/E12.yaml
!PYTHONPATH=. python scripts/eval_test.py --config configs/experiments/E13_rrctx.yaml
import os
if os.path.exists('data/processed/mitbih_incartII/X_beats.npy'):
    !PYTHONPATH=. python scripts/eval_test.py --config configs/experiments/E14_incII.yaml

## 14. E15 — agregar SVDB (un solo cambio sobre E14)
**Sesión nueva:** corre 1, 2, 3, 4 y la **primera celda de la 11** (enlaza los datos ya construidos en Drive). Luego esta sección y al final la 10 para subir.

Los folds de los 66 sujetos actuales quedan fijos (`configs/cv_folds_E14.json`), así que la comparación con E14 es fold a fold. ~1–1.5 h en GPU (CV) + ~15 min (test).

In [ ]:
import os
for d in ['data/processed/mitbih_incartII_svdb']:
    target = f'{RUNS}/{d}'
    os.makedirs(target, exist_ok=True)
    if not os.path.exists(d):
        os.symlink(target, d)
if not os.path.exists('data/processed/mitbih_incartII_svdb/X_beats.npy'):
    !PYTHONPATH=. python scripts/build_extra_db.py --merge data/processed/mitbih_incartII data/processed/svdb_ch0 --out data/processed/mitbih_incartII_svdb
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E15_svdb.yaml
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E15_svdb
!PYTHONPATH=. python scripts/eval_test.py --config configs/experiments/E15_svdb.yaml

## 15. E16 — focal loss α_SVEB 8 → 3 (un solo cambio sobre E15) + test de E15 con su umbral
**Sesión nueva:** corre 1, 2, 3, 4, la primera celda de la 11 y la primera mitad de la 14 (los `symlink`), o simplemente esta celda (enlaza lo necesario). Luego la 10 para subir. ~1.5 h en GPU.

In [ ]:
import os
RUNS = '/content/drive/MyDrive/heartlype_runs'
for d in ['data/processed/inc_ch1', 'data/processed/mitbih_incartII', 'data/processed/svdb_ch0', 'data/processed/mitbih_incartII_svdb']:
    target = f'{RUNS}/{d}'
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); os.symlink(target, d)
assert os.path.exists('data/processed/mitbih_incartII_svdb/X_beats.npy'), 'falta correr la sección 14 una vez'
# test de E15 con su umbral 0.54 (reutiliza los modelos ya entrenados en Drive, no reentrena)
!PYTHONPATH=. python scripts/eval_test.py --config configs/experiments/E15_svdb.yaml
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E16_alpha3.yaml
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E16_alpha3
!PYTHONPATH=. python scripts/eval_test.py --config configs/experiments/E16_alpha3.yaml

## 10. Devolver los resultados a GitHub (rama `colab-results`)
Necesitas un **token de GitHub** (es como una contraseña temporal solo para este repo):
1. Entra a https://github.com/settings/personal-access-tokens/new
2. *Token name*: `colab`; *Expiration*: 7 días.
3. *Repository access* → **Only select repositories** → `ecg-arrhythmia-thesis`.
4. *Permissions* → *Repository permissions* → **Contents: Read and write**.
5. **Generate token**, cópialo (empieza con `github_pat_`) y pégalo cuando esta celda lo pida.

No lo pegues en el chat ni en el código: la celda lo pide de forma oculta y no queda guardado. Cuando terminemos puedes borrarlo en esa misma página.

In [ ]:
from getpass import getpass
token = getpass('GitHub token: ')
!git config user.email "csifuentes398@gmail.com" && git config user.name "ClaudiaSifuentes"
!git checkout -q -B colab-results
!mkdir -p results_colab && cp -rL reports/cv reports/oof reports/thresholds reports/test results_colab/ 2>/dev/null; cp configs/incart_subjects.json results_colab/
!find results_colab -name 'part_*.npz' -delete
!git add -f results_colab configs/incart_subjects.json && git commit -qm "results: Colab CV runs" && git push -q -f https://{token}@github.com/ClaudiaSifuentes/ecg-arrhythmia-thesis.git colab-results
print('listo: rama colab-results')